In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

In [ ]:
# Confirm data time series overlap with 12442.  If data time series is the same then delete, if not then concatenate with 12442 into the Additional PRPA station (ENV Native ID 547) and delete

# Confirm data time series overlap with 12454, if the same then delete, if gap fill the concatenate into 12454, and then delete.

# Confirm data time series overlap with 12568.  If data time series is the same then delete 12568, if not then concatenate with 12568 into the Additional PRPA station (ENV Native ID 547) and delete.


In [5]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
"Concatenate data with 13217, then delete",
"Concatenate with 12283 and delete",
"Concatenate with 12441 and delete",
"Concatenate with Station ID 12589 if appropriate, and then delete.",
"Data needs concatenating with 12170",
"Data should be concatenated into one record",
"Confirm data time series overlap with 12454, if the same then delete, if gap fill the concatenate into 12454, and then delete."

]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

len(df_concat)

df_concat =  df_concat[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)
df_concat["Network ID"] = pd.to_numeric(df_concat["Network ID"], errors="coerce").astype("Int64")
df_concat["Station ID"] = pd.to_numeric(df_concat["Station ID"], errors="coerce").astype("Int64")
df_concat

# Split on '->'
split_ids = df_concat['Native ID'].astype(str).str.split('->', expand=True)

df_concat['old_native_id'] = split_ids[0].str.strip()
df_concat['new_native_id'] = split_ids[1].str.strip()
df_concat = df_concat.drop(columns=['Native ID'])

df_concat = df_concat.rename(columns={
    'Station ID': 'old_station_id',
    'Unique Names': 'old_station_name'
})

df_concat


,ISSUE,old_station_id,Network ID,Network Name,old_station_name,old_native_id,new_native_id
0,"Concatenate data with 13217, then delete",13128,5,BCH-GSO-HMP,NaN,JHL,None
1,Concatenate with 12283 and delete,2613,9,BC-Air,Smithers St Josephs,M107005,171
2,Concatenate with 12441 and delete,12572,9,BC-Air,NA -> Smithers Muheim Memorial,Smithers Muheim Memorial,563
3,Concatenate with Station ID 12589 if appropria...,2630,9,BC-Air -> MVRD-AIR,Abbotsford Central,E238212,T033
4,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,NA -> Crofton Elementary,Crofton Elementary,564
5,Data needs concatenating with 12170,1580,9,BC-Air,Kitimat Whitesail,M106010,80
6,Data should be concatenated into one record,2601,9,BC-Air,Prince George Plaza 400,M109912,144


In [6]:
query = """
SELECT station_id
FROM meta_station
WHERE native_id = %s
  AND network_id = %s
"""
def get_station_id(native_id, network_id, engine):
    df = pd.read_sql(
        query,
        engine,
        params=(str(native_id), int(network_id))
    )

    if df.empty:
        return None

    return int(df.iloc[0]["station_id"])

df_concat["new_station_id_from_n"] = df_concat.apply(
    lambda row: get_station_id(
        row["new_native_id"],
        row["Network ID"],
        engine
    ),
    axis=1
)

df_concat["new_station_id_from_n"] = (
    pd.to_numeric(df_concat["new_station_id_from_n"], errors="coerce")
      .astype("Int64")
)
df_concat

,ISSUE,old_station_id,Network ID,Network Name,old_station_name,old_native_id,new_native_id,new_station_id_from_n
0,"Concatenate data with 13217, then delete",13128,5,BCH-GSO-HMP,NaN,JHL,None,<NA>
1,Concatenate with 12283 and delete,2613,9,BC-Air,Smithers St Josephs,M107005,171,12283
2,Concatenate with 12441 and delete,12572,9,BC-Air,NA -> Smithers Muheim Memorial,Smithers Muheim Memorial,563,12441
3,Concatenate with Station ID 12589 if appropria...,2630,9,BC-Air -> MVRD-AIR,Abbotsford Central,E238212,T033,<NA>
4,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,NA -> Crofton Elementary,Crofton Elementary,564,<NA>
5,Data needs concatenating with 12170,1580,9,BC-Air,Kitimat Whitesail,M106010,80,12185
6,Data should be concatenated into one record,2601,9,BC-Air,Prince George Plaza 400,M109912,144,12254


In [7]:
df_concat

,ISSUE,old_station_id,Network ID,Network Name,old_station_name,old_native_id,new_native_id,new_station_id_from_n
0,"Concatenate data with 13217, then delete",13128,5,BCH-GSO-HMP,NaN,JHL,None,<NA>
1,Concatenate with 12283 and delete,2613,9,BC-Air,Smithers St Josephs,M107005,171,12283
2,Concatenate with 12441 and delete,12572,9,BC-Air,NA -> Smithers Muheim Memorial,Smithers Muheim Memorial,563,12441
3,Concatenate with Station ID 12589 if appropria...,2630,9,BC-Air -> MVRD-AIR,Abbotsford Central,E238212,T033,<NA>
4,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,NA -> Crofton Elementary,Crofton Elementary,564,<NA>
5,Data needs concatenating with 12170,1580,9,BC-Air,Kitimat Whitesail,M106010,80,12185
6,Data should be concatenated into one record,2601,9,BC-Air,Prince George Plaza 400,M109912,144,12254


In [8]:
query = """
SELECT native_id
FROM meta_station
WHERE station_id = %s
  AND network_id = %s
"""

def get_new_native_id_from_s(concat_station_id, network_id, engine):
    # handle <NA> safely
    if pd.isna(concat_station_id):
        return None

    df = pd.read_sql(
        query,
        engine,
        params=(int(concat_station_id), int(network_id))
    )

    if df.empty:
        return None

    return df.iloc[0]["native_id"]  # TEXT → string

df_concat["concat_station_id"] = (
    df_concat["ISSUE"]
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")  # nullable integer
)


df_concat.loc[
    df_concat["new_native_id"] == "144",
    "concat_station_id"
] = df_concat.loc[
    df_concat["new_native_id"] == "144",
    "new_station_id_from_n"
]



df_concat["new_native_id_from_s"] = df_concat.apply(
    lambda row: get_new_native_id_from_s(
        row["concat_station_id"],
        row["Network ID"],
        engine
    ),
    axis=1
)
df_concat


,ISSUE,old_station_id,Network ID,Network Name,old_station_name,old_native_id,new_native_id,new_station_id_from_n,concat_station_id,new_native_id_from_s
0,"Concatenate data with 13217, then delete",13128,5,BCH-GSO-HMP,NaN,JHL,None,<NA>,13217,JHL
1,Concatenate with 12283 and delete,2613,9,BC-Air,Smithers St Josephs,M107005,171,12283,12283,171
2,Concatenate with 12441 and delete,12572,9,BC-Air,NA -> Smithers Muheim Memorial,Smithers Muheim Memorial,563,12441,12441,563
3,Concatenate with Station ID 12589 if appropria...,2630,9,BC-Air -> MVRD-AIR,Abbotsford Central,E238212,T033,<NA>,12589,Abbotsford Central
4,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,NA -> Crofton Elementary,Crofton Elementary,564,<NA>,12454,E316070
5,Data needs concatenating with 12170,1580,9,BC-Air,Kitimat Whitesail,M106010,80,12185,12170,102
6,Data should be concatenated into one record,2601,9,BC-Air,Prince George Plaza 400,M109912,144,12254,12254,144


In [9]:
df_work = df_concat.iloc[[0, 1, 2, 5, 6]].drop(columns = ['new_native_id', 'new_station_id_from_n'])
df_work



,ISSUE,old_station_id,Network ID,Network Name,old_station_name,old_native_id,concat_station_id,new_native_id_from_s
0,"Concatenate data with 13217, then delete",13128,5,BCH-GSO-HMP,NaN,JHL,13217,JHL
1,Concatenate with 12283 and delete,2613,9,BC-Air,Smithers St Josephs,M107005,12283,171
2,Concatenate with 12441 and delete,12572,9,BC-Air,NA -> Smithers Muheim Memorial,Smithers Muheim Memorial,12441,563
5,Data needs concatenating with 12170,1580,9,BC-Air,Kitimat Whitesail,M106010,12170,102
6,Data should be concatenated into one record,2601,9,BC-Air,Prince George Plaza 400,M109912,12254,144


### 5, seems first concat, then revise old_native_id to "80"

In [10]:
query = """
SELECT native_id
FROM meta_station
WHERE station_id = %s
  AND network_id = %s
"""

def get_old_native_id_from_s(concat_station_id, network_id, engine):
    # handle <NA> safely
    if pd.isna(concat_station_id):
        return None

    df = pd.read_sql(
        query,
        engine,
        params=(int(concat_station_id), int(network_id))
    )

    if df.empty:
        return None

    return df.iloc[0]["native_id"]  # TEXT → string

df_concat.apply(
    lambda row: get_new_native_id_from_s(
        row["old_station_id"],
        row["Network ID"],
        engine
    ),
    axis=1
)

0                         JHL
1                     M107005
2    Smithers Muheim Memorial
3                     E238212
4          Crofton Elementary
5                     M106010
6                     M109912
dtype: object

In [14]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id  = %s
       AND s.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id  = %s
       AND s.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id  = %s
           AND s.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id  = %s
           AND s.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE s_old.native_id  = %s
       AND s_old.station_id = %s
       AND s_new.native_id  = %s
       AND s_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_native_id,
    old_station_id,
    new_native_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_native_id, old_station_id,

        # n_new
        new_native_id, new_station_id,

        # n_overlap
        old_native_id, old_station_id,
        new_native_id, new_station_id,

        # n_overlap_same_datum
        old_native_id, old_station_id,
        new_native_id, new_station_id
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = df_work.apply(
    lambda r: count_station_stats(
        r["old_native_id"],
        r["old_station_id"],
        r["new_native_id_from_s"],
        r["concat_station_id"],
        engine
    ),
    axis=1
)

df_work[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

In [15]:
df_work

,ISSUE,old_station_id,Network ID,Network Name,old_station_name,old_native_id,concat_station_id,new_native_id_from_s,n_old,n_new,n_overlap,n_overlap_same_datum
0,"Concatenate data with 13217, then delete",13128,5,BCH-GSO-HMP,NaN,JHL,13217,JHL,151279,195020,19083,17867
1,Concatenate with 12283 and delete,2613,9,BC-Air,Smithers St Josephs,M107005,12283,171,380798,264102,1550,34
2,Concatenate with 12441 and delete,12572,9,BC-Air,NA -> Smithers Muheim Memorial,Smithers Muheim Memorial,12441,563,5664,156497,1142,1142
5,Data needs concatenating with 12170,1580,9,BC-Air,Kitimat Whitesail,M106010,12170,102,318191,759781,316648,617
6,Data should be concatenated into one record,2601,9,BC-Air,Prince George Plaza 400,M109912,12254,144,595855,429239,4322,71


In [13]:
### better to use both native_id and station_id to constrain

In [16]:
from sqlalchemy import text

SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id  = :new_native_id
  AND s.station_id = :new_station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_MOVE = text("""
WITH updated AS (
    UPDATE obs_raw o
    SET history_id = :new_hist_id
    FROM meta_history h_old
    JOIN meta_station s_old ON h_old.station_id = s_old.station_id
    WHERE o.history_id = h_old.history_id
      AND s_old.native_id  = :old_native_id
      AND s_old.station_id = :old_station_id
      AND NOT EXISTS (
          SELECT 1
          FROM obs_raw o_new
          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
          JOIN meta_station s_new ON h_new.station_id = s_new.station_id
          WHERE s_new.native_id  = :new_native_id
            AND s_new.station_id = :new_station_id
            AND o_new.obs_time = o.obs_time
            AND o_new.vars_id  = o.vars_id
      )
    RETURNING 1
)
SELECT COUNT(*) FROM updated;
""")

def move_station_obs_fast(
    old_native_id,
    old_station_id,
    new_native_id,
    new_station_id,
    engine
):
    with engine.begin() as conn:
        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {
                "new_native_id": new_native_id,
                "new_station_id": new_station_id,
            }
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_native_id}/{new_station_id} has no history, skipping.")
            return 0

        n_move = conn.execute(
            SQL_MOVE,
            {
                "old_native_id": old_native_id,
                "old_station_id": old_station_id,
                "new_native_id": new_native_id,
                "new_station_id": new_station_id,
                "new_hist_id": new_hist_id,
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from {old_native_id}/{old_station_id} "
            f"→ {new_native_id}/{new_station_id}"
        )

        return n_move

results = []

for i, row in df_work.iterrows():
    n = move_station_obs_fast(
        row["old_native_id"],
        row["old_station_id"],
        row["new_native_id_from_s"],
        row["concat_station_id"],
        engine
    )
    results.append(n)

df_work["n_moved"] = results

Moved 132196 rows from JHL/13128 → JHL/13217
Moved 379248 rows from M107005/2613 → 171/12283
Moved 4522 rows from Smithers Muheim Memorial/12572 → 563/12441
Moved 1543 rows from M106010/1580 → 102/12170
Moved 591533 rows from M109912/2601 → 144/12254


### Delete

In [22]:
station_pairs = (
    df_work[["old_station_id", "old_native_id"]]
    .dropna()
    .assign(
        old_station_id=lambda d: d["old_station_id"].astype(int),
        old_native_id=lambda d: d["old_native_id"].astype(str),
    )
    .drop_duplicates()
    .to_records(index=False)
)

station_pairs

rec.array([(13128, 'JHL'), ( 2613, 'M107005'),
           (12572, 'Smithers Muheim Memorial'), ( 1580, 'M106010'),
           ( 2601, 'M109912')],
          dtype=[('old_station_id', '<i8'), ('old_native_id', 'O')])

In [23]:
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o
JOIN meta_history h ON o.history_id = h.history_id
JOIN meta_station s ON h.station_id = s.station_id
WHERE fr.obs_raw_id = o.obs_raw_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE dv.history_id = h.history_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_history = sa.text("""
DELETE FROM meta_history h
USING meta_station s
WHERE h.station_id = s.station_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
  AND native_id  = :native_id
""")

with engine.begin() as conn:
    for station_id, native_id in station_pairs:

        station_id = int(station_id)     # convert np.int64 → int
        native_id  = str(native_id)      # (safe, optional)

        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": station_id, "native_id": native_id}
        )

        res_obs = conn.execute(
            delete_obs,
            {"station_id": station_id, "native_id": native_id}
        )

        res_dv = conn.execute(
            delete_obs_derived,
            {"station_id": station_id, "native_id": native_id}
        )

        res_hist = conn.execute(
            delete_history,
            {"station_id": station_id, "native_id": native_id}
        )

        res_sta = conn.execute(
            delete_station,
            {"station_id": station_id, "native_id": native_id}
        )

        print(
            f"Station {station_id}/{native_id}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Station 13128/JHL: flags=0, obs_raw=19083, obs_derived=0, meta_history=1, meta_station=1
Station 2613/M107005: flags=0, obs_raw=1550, obs_derived=24, meta_history=1, meta_station=1
Station 12572/Smithers Muheim Memorial: flags=0, obs_raw=1142, obs_derived=0, meta_history=1, meta_station=1
Station 1580/M106010: flags=0, obs_raw=316648, obs_derived=24, meta_history=1, meta_station=1
Station 2601/M109912: flags=0, obs_raw=4322, obs_derived=0, meta_history=1, meta_station=1


In [ ]:
Only overlapping records remain in obs_raw; all non-overlapping records have been relocated. Therefore, the overlapping records will be deleted to allow complete removal of the station.

### Consider Deleting - data is not worth keeping 12280

In [25]:
from tqdm import tqdm
import sqlalchemy as sa


# List of station_ids to delete
station_ids_to_delete = [12280]  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations:   0%|          | 0/1 [00:00<?, ?it/s]

Deleting stations: 100%|██████████| 1/1 [01:28<00:00, 88.15s/it]

Station 12280: flags=0, obs_raw=94487, obs_derived=0, meta_history=1, meta_station=1
